Case Study : Business Requirement (Real-Time Scenario)
A retail company receives a daily sales_data.csv file from its source system. The file contains order-level transaction data including order ID, customer name, product, category, order amount, and order date.
The Data Engineering team must build an ETL pipeline that:
1.	Reads the CSV file from the landing zone.
2.	Validates and cleans the data.
3.	Removes duplicate records based on order_id (keeping the latest order_date).
4.	Filters out invalid records (amount <= 0).
5.	Adds derived columns:
a.	tax_amount (18% GST)
b.	total_amount (amount + tax)
6.	Generates category-level aggregated metrics:
a.	Total Revenue
b.	Total Orders
7.	Stores:
a.	Cleaned data in Silver layer
b.	Aggregated data in Gold layer
8.	Final output should be stored in Delta format (production-ready) and optionally in Parquet.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

df=spark.read.format('csv')\
            .option('inferSchema',True)\
            .option('header',True)\
            .load('/Volumes/workspace/default/abhi_volume_1/big_practice_dataset.csv')            

df=df.withColumn('order_date',date_format(col('order_date'),'dd-MM-yyyy'))

df_clean=df.filter(col('amount')>0)\
    .dropna(subset=['order_id','customer_id','amount'])\
    .drop_duplicates(subset=['order_id'])        


df_clean=df_clean.withColumn('tax_amount',col('amount')*0.18)\
                 .withColumn('total_amount',col('amount')+col('tax_amount'))   
df_clean.display()


df_clean.createOrReplaceTempView('orders_data')
df_agg=spark.sql("""select product,sum(total_amount)as total_amount,avg(total_amount)as avg_amount from orders_data group by product """)

df_agg.write.mode('overwrite')\
            .format('delta')\
            .saveAsTable('default.orders_tbl')        

df_agg.display()


